## Landslide event mapping using Sentinel-2 data

Map the extent of landslides using Sentinel-2 data.

Generate a pre event mosaic and use the first available post event image. 
Calculate the NDVI for both and generate a change image using an NDVI threshold and the formula (pre-image * 10) + post-image to generate the following classes for each image:

- 00 indicates no change
- 01 change
- 10 is not reasonable because it means that pre-event is true and post not true and can be disregarded
- 11 both images are true --> in both images NDVI is below the threshold and no change happened

In [44]:
import openeo
from openeo.processes import quantiles
import leafmap
from shapely.geometry import shape
from folium.plugins import Draw
from IPython.display import JSON
import urllib
import json
import numpy as np
import pandas as pd

In [ ]:
# if package is not installed, run the following command:
#pip install leafmap

In [45]:
# define properties for the outputs
from pathlib import Path

out_dir = Path("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/landslide/nepal")
mkdir = out_dir.mkdir(parents=True, exist_ok=True)

event = "Nepal_2026_mosaics"

BANDS = ["B02", "B03", "B04", "B08", "SCL"]  

### 1) Define time frame and extent

In [46]:
# 1 . Define the time periods for pre- and post event date
# blatten

#PRE_DATE  = ("2024-05-01", "2024-06-30")   
#POST_DATE = ("2025-05-28", "2025-05-31")   

# Pre and post date merano
#PRE_DATE  = ("2026-06-01", "2026-06-28")   
#POST_DATE = ("2026-06-29", "2026-07-06")  

# vals
#PRE_DATE  = ("2026-07-01", "2026-07-30")   
#POST_DATE = ("2026-08-08", "2026-08-15")   

# Ecrins
#PRE_DATE  = ("2024-06-01", "2024-06-20")   
#POST_DATE = ("2024-06-25", "2024-07-05")

# Nepal
PRE_DATE  = ("2026-07-01", "2026-08-20")   
POST_DATE = ("2026-08-26", "2026-09-10")

In [47]:
# open a map and zoom to the area of interest
m = leafmap.Map(center=(46.65, 11.4), zoom=8.5)
m

Map(center=[46.65, 11.4], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_ou…

In [48]:
feat = m.draw_features
geom_dict = feat[0]['geometry']
geom = shape(geom_dict)

minx, miny, maxx, maxy = geom.bounds

bbox = {
    "west": minx,
    "south": miny,
    "east": maxx,
    "north": maxy,
}

print(bbox)

{'west': 85.264893, 'south': 28.14466, 'east': 85.594482, 'north': 28.399857}


### 2) Authentificate and load the pre and post Sentinel-2 cube

In [49]:
# load sentinel-2 data after the flood event
connection = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()
connection.authenticate_oidc()

# Load Sentinel-2 data
def load_s2(temporal_extent):
    cube = connection.load_collection(
        "SENTINEL2_L2A",
        spatial_extent=bbox,
        temporal_extent=list(temporal_extent),
        bands=BANDS,
        max_cloud_cover=50,
    )
    return cube

pre_cube=load_s2(PRE_DATE)
post_cube = load_s2(POST_DATE)

Authenticated using refresh token.
Authenticated using refresh token.


### 3) Create the mask from SCL

Create the mask using the following SCL values:

```
1  = SC_SATURATED_DEFECTIVE
3  = SC_CLOUD_SHADOW
7  = SC_CLOUD_LOW_PROBA / UNCLASSIFIED
8  = SC_CLOUD_MEDIUM_PROBA
9  = SC_CLOUD_HIGH_PROBA
10 = SC_THIN_CIRRUS
```

In [50]:
def mask_clouds(cube):
    scl = cube.band("SCL")
    return (
    (scl == 1) |
    (scl == 3) |
    (scl == 7) |
    (scl == 8) |
    (scl == 9) |
    (scl == 10)
    )

### 4) Generate the layer of valid observations

In [51]:
post_cube_mask = mask_clouds(post_cube)
post_cube_masked = post_cube.mask(post_cube_mask).mean_time()

In [52]:
file_post_cube = out_dir/ f"post_{event}.tif"

print(file_post_cube)

#post_cube_masked.download(file_post_cube, format='GTiff')

/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/landslide/nepal/post_Nepal_2026_mosaics.tif


In [53]:
job = post_cube_masked.create_job(title=f"post_{event}", description=f"post_{event}")
job.start_and_wait()

0:00:00 Job 'j-260910115521494a9c7c631dc2822857': send 'start'
0:00:06 Job 'j-260910115521494a9c7c631dc2822857': queued (progress 0%)
0:00:11 Job 'j-260910115521494a9c7c631dc2822857': queued (progress 0%)
0:00:18 Job 'j-260910115521494a9c7c631dc2822857': queued (progress 0%)
0:00:26 Job 'j-260910115521494a9c7c631dc2822857': queued (progress 0%)
0:00:36 Job 'j-260910115521494a9c7c631dc2822857': queued (progress 0%)
0:00:48 Job 'j-260910115521494a9c7c631dc2822857': queued (progress 0%)
0:01:04 Job 'j-260910115521494a9c7c631dc2822857': running (progress N/A)
0:01:23 Job 'j-260910115521494a9c7c631dc2822857': running (progress N/A)
0:01:47 Job 'j-260910115521494a9c7c631dc2822857': running (progress N/A)
0:02:17 Job 'j-260910115521494a9c7c631dc2822857': running (progress N/A)
0:02:54 Job 'j-260910115521494a9c7c631dc2822857': running (progress N/A)
0:03:41 Job 'j-260910115521494a9c7c631dc2822857': running (progress N/A)
0:04:40 Job 'j-260910115521494a9c7c631dc2822857': error (progress N/A)
Yo

JobFailedException: Batch job 'j-260910115521494a9c7c631dc2822857' didn't finish successfully. Status: error (after 0:04:40).

In [12]:
job.download_results(file_post_cube)

{PosixPath('/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/landslide/ecrins/post_Ecrins_2024_mosaics.tif'): {'eo:bands': [{'name': 'B02'},
   {'name': 'B03'},
   {'name': 'B04'},
   {'name': 'B08'},
   {'name': 'SCL'}],
  'href': 'https://s3.waw3-1.openeo.v1.dataspace.copernicus.eu/openeo-data-prod-waw4-1/batch_jobs/j-26090910172945658132dbdc18fc899e/openEO.tif?X-Proxy-Head-As-Get=true&X-Amz-Algorithm=AWS4-HMAC-SHA256&X-Amz-Credential=bf81f1518fed431195682b62fb490b90%2F20260909%2Fwaw4-1%2Fs3%2Faws4_request&X-Amz-Date=20260909T121913Z&X-Amz-Expires=86400&X-Amz-SignedHeaders=host&X-Amz-Security-Token=eyJhbGciOiJSUzI1NiIsInR5cCI6IkpXVCJ9.eyJyb2xlX2FybiI6ImFybjpvcGVuZW93czppYW06Ojpyb2xlL29wZW5lby1kYXRhLXByb2Qtd2F3NC0xLXdvcmtzcGFjZSIsImluaXRpYWxfaXNzdWVyIjoib3BlbmVvLnByb2Qud2F3My0xLm9wZW5lby1pbnQudjEuZGF0YXNwYWNlLmNvcGVybmljdXMuZXUiLCJodHRwczovL2F3cy5hbWF6b24uY29tL3RhZ3MiOnsicHJpbmNpcGFsX3RhZ3MiOnsiam9iX2lkIjpbImotMjYwOTA5MTAxNzI5NDU2NTgxMzJkYmRjMThmYzg5OWUiXSwidXNlcl9pZCI6WyI5N2JmZDdmMy1kMTQ1LTRlMDE

### 5) Generate monthly mosaic for pre event 

In [40]:
pre_cube_mask = mask_clouds(pre_cube)

pre_cube_masked = pre_cube.mask(pre_cube_mask).reduce_dimension(
    dimension='t',
    reducer=lambda x: quantiles(data=x, probabilities=[0.25])
)

In [41]:
file_pre_cube = out_dir/ f"pre_mosaic_{event}.tif"

print(file_pre_cube)

#pre_cube_masked.download(file_pre_cube, format='GTiff')

/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/landslide/ecrins/pre_mosaic_Ecrins_2024_mosaics.tif


### 6) Calculate NDVI for pre and post event

In [42]:
# calcualte NDVI for post-flood period
def compute_ndvi(cube):
    nir = cube.band("B08")
    red = cube.band("B04")
    return (nir - red) / (nir + red)

post_ndvi = compute_ndvi(post_cube_masked)
pre_ndvi = compute_ndvi(pre_cube_masked)

In [16]:
file_pre_ndvi = out_dir/ f"pre_mosaic_{event}_NDVI.tif"
file_post_ndvi = out_dir/ f"post_{event}_NDVI.tif"

pre_ndvi.download(file_pre_ndvi)
post_ndvi.download(file_post_ndvi)

### 7) Calculate Change between pre and post image

In [43]:
# extract treshold + apply to each image and then classify image
thresh = 0.3

pre_ndvi_thresh  = (pre_ndvi  <= thresh).apply(lambda x: x * 1)  
post_ndvi_thresh = (post_ndvi <= thresh).apply(lambda x: x * 1)   

change_ndvi = (pre_ndvi_thresh * 10) + post_ndvi_thresh

# export
file_change_ndvi = out_dir/ f"change_{event}_NDVI.tif"
print(file_change_ndvi)
#change_ndvi.download(file_change_ndvi)

/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/landslide/ecrins/change_Ecrins_2024_mosaics_NDVI.tif


# Calculate coherence for pre and post event

Compare the above calculated results with Sentinel-1 coherence 

Coherence can be calculated for the period before the event and after the event seperatedly or between images before the event and the state after. Just change the dates below

### 1) Check out the bursts for the area

In [ ]:
PRE_DATE  = ("2026-07-01", "2026-08-15")   
POST_DATE = ("2026-08-08", "2026-08-15")   

In [ ]:
https_request = f"https://catalogue.dataspace.copernicus.eu/odata/v1/Bursts?$filter=" + urllib.parse.quote(
    f"ContentDate/Start ge {PRE_DATE[0]}T00:00:00.000Z and ContentDate/Start le {POST_DATE[1]}T23:59:59.000Z and "
    f"PolarisationChannels eq 'VV' and "
    f"OData.CSC.Intersects(area=geography'SRID=4326;{geom.wkt}')"
) + "&$top=1000"

with urllib.request.urlopen(https_request) as response:
    content = response.read().decode()
bursts = json.loads(content)

In [ ]:
bursts_uniqueTrack = {}
burstId_list = []
track_list = []
for b in bursts['value']:
    if b['RelativeOrbitNumber'] not in track_list:
        bursts_uniqueTrack[b['RelativeOrbitNumber']] = {}
        track_list.append(b['RelativeOrbitNumber'])
    burstId_subswath = f"BurstId: {b['BurstId']}, {b['SwathIdentifier']}"
    if burstId_subswath not in burstId_list:
        bursts_uniqueTrack[b['RelativeOrbitNumber']][burstId_subswath] = b['GeoFootprint']['coordinates']
        burstId_list.append(burstId_subswath)

In [ ]:
burstId_list

### 2) Calculate coherence

Calculate coherence either for
- pre-event 
- post-event 

... and calculate difference

or for
- pre-event until post-event to see the difference in coherence

In [ ]:
url = "https://openeo.dataspace.copernicus.eu"
connection = openeo.connect(url).authenticate_oidc()

def s1_coherence(temporal_extent):
    datacube = connection.datacube_from_process(
        process_id="sentinel1_sar_coherence",
        namespace="https://raw.githubusercontent.com/ESA-APEx/apex_algorithms/refs/heads/main/algorithm_catalog/eurac/sentinel1_sar_coherence/openeo_udp/sentinel1_sar_coherence.json",
        temporal_extent=(temporal_extent),
        temporal_baseline=12,
        burst_id=92639,
        sub_swath="IW1",
    
        polarization="VV"
    )
    return datacube

pre_cube = s1_coherence(PRE_DATE).save_result(format="GTiff")
#post_cube = s1_coherence(POST_DATE).save_result(format="GTiff")

In [ ]:
pre_job = pre_cube.create_job(title="event coherence")
#post_job = post_cube.create_job(title="Post-event coherence")

pre_job.start_and_wait()
#post_job.start_and_wait()

In [ ]:
pre_job.download_results("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/landslide/coherence_pre_post_vals")
#post_job.download_results("/mnt/CEPH_PROJECTS/provinzBZ_risk_EO/landslide/coherence_post_blatten")